# Kubeflow SDK: Orchestrated ML Pipeline Tutorial

This tutorial demonstrates how to use the `PipelinesClient` to orchestrate a multi-component ML pipeline chaining:

1. **Data Preprocessing (`SparkClient`)**: Clean and prepare datasets using Spark.
2. **Hyperparameter Tuning (`OptimizerClient`)**: Search for the best training parameters using Katib.
3. **Final Model Training (`TrainerClient`)**: Train the model with the optimal parameters.
4. **Model Registration (`ModelRegistryClient`)**: Register the trained model in the Kubeflow Model Registry.

We use Kubeflow Pipelines (KFP) as our orchestrator, chaining these steps in a Directed Acyclic Graph (DAG).

## Step 1: Initialize Clients and Setup

Import the required SDK modules. If KFP is not available in the execution context, we will construct a mock client to dry-run the pipeline components locally.

In [ ]:
import os
from kubeflow.pipelines import PipelinesClient, dsl

print("Pipelines module loaded successfully!")

## Step 2: Define Pipeline Components

We define the individual tasks as `@dsl.component` steps. Each component imports and runs the respective Kubeflow SDK client internally.

### Component 1: Data Preprocessing (Spark)

This component prepares raw data using the `SparkClient`.

In [ ]:
@dsl.component
def preprocess_data(input_path: str, output_path: str):
    import os

    print(f"Reading raw data from {input_path}")

    try:
        from kubeflow.spark import SparkClient

        client = SparkClient()
        print("Connecting to Spark Connect cluster...")
        spark = client.connect()
        print("Connected successfully!")
        # Simulate Spark data cleaning
        df = spark.read.parquet(input_path)
        df_clean = df.dropna()
        df_clean.write.parquet(output_path)
        spark.stop()
    except Exception as e:
        print(
            f"Spark client failed or not available ({e}). Simulating local preprocessing fallback."
        )
        os.makedirs(output_path, exist_ok=True)
        with open(os.path.join(output_path, "cleaned_data.txt"), "w") as f:
            f.write("Simulated Spark-preprocessed dataset")

    print(f"Data preprocessing completed. Results saved to {output_path}")

### Component 2: Hyperparameter Tuning (Katib)

This component launches a Katib hyperparameter search space using `OptimizerClient` to identify the optimal learning rate.

In [ ]:
@dsl.component
def tune_hyperparameters(data_path: str) -> str:
    print(f"Tuning hyperparameters using processed dataset from {data_path}")

    # Write trial training script to a file to support source serialization in nested function context
    with open("trial_script.py", "w") as f:
        f.write('def trial_train(lr: float):\n    loss = 0.5 / lr\n    print(f"loss={loss:.4f}")\n')

    import trial_script

    try:
        from kubeflow.trainer import TrainJobTemplate, CustomTrainer
        from kubeflow.optimizer import OptimizerClient, Search, TrialConfig, Objective, Direction

        optimizer_client = OptimizerClient()
        trial_template = TrainJobTemplate(
            trainer=CustomTrainer(func=trial_script.trial_train, func_args={"lr": 0.01})
        )
        search_space = {"lr": Search.uniform(min=0.001, max=0.05)}
        objectives = [Objective(metric="loss", direction=Direction.MINIMIZE)]
        trial_config = TrialConfig(num_trials=2, parallel_trials=1)

        opt_job_name = optimizer_client.optimize(
            trial_template=trial_template,
            search_space=search_space,
            objectives=objectives,
            trial_config=trial_config,
        )
        optimizer_client.wait_for_job_status(opt_job_name)
        best_results = optimizer_client.get_best_results(opt_job_name)
        best_lr = (
            best_results.parameters["lr"]
            if best_results
            and hasattr(best_results, "parameters")
            and "lr" in best_results.parameters
            else "0.01"
        )
    except Exception as e:
        print(
            f"OptimizerClient/Katib not available ({e}). Falling back to selected hyperparameter."
        )
        best_lr = "0.012"

    print(f"Best learning rate selected: {best_lr}")
    return best_lr

### Component 3: PyTorch Model Training (Trainer)

This component uses `TrainerClient` to train our final model using the best learning rate.

In [ ]:
@dsl.component
def train_final_model(best_lr: str, data_path: str, trained_model_uri: dsl.OutputPath(str)):
    import os

    print(f"Training final model with lr={best_lr} using dataset at {data_path}")

    # Write final training script to a file to support source serialization in nested function context
    with open("final_script.py", "w") as f:
        f.write('def final_train(lr: float):\n    print(f"Training with lr={lr}")\n')

    import final_script

    try:
        from kubeflow.trainer import TrainerClient, CustomTrainer

        trainer_client = TrainerClient()
        job_name = trainer_client.train(
            trainer=CustomTrainer(func=final_script.final_train, func_args={"lr": float(best_lr)})
        )
        trainer_client.wait_for_job_status(job_name)
    except Exception as e:
        print(f"TrainerClient failed or not available ({e}). Simulating local model compilation.")

    os.makedirs(os.path.dirname(trained_model_uri), exist_ok=True)
    with open(trained_model_uri, "w") as f:
        f.write("s3://my-org-models/orchestrated-model/checkpoint")
    print(f"Trained model URI saved to output parameter: {trained_model_uri}")

### Component 4: Model Registration (Model Registry)

This component registers the resulting model version in the Model Registry.

In [ ]:
@dsl.component
def register_model(trained_model_uri: str, model_name: str, version: str):
    import os
    from unittest.mock import MagicMock

    print(f"Registering model '{model_name}' version '{version}' from '{trained_model_uri}'")
    mr_host = os.environ.get(
        "MODEL_REGISTRY_URL", "http://model-registry-service.kubeflow.svc.cluster.local:8080"
    )

    try:
        from kubeflow.hub import ModelRegistryClient

        mr_client = ModelRegistryClient(base_url=mr_host)
        list(mr_client.list_models())
        is_mock = False
    except Exception as e:
        print(f"ModelRegistryClient not available ({e}). Falling back to mock registration.")
        is_mock = True

    if is_mock:

        class MockModelRegistryClient:
            def register_model(
                self,
                name,
                uri,
                version,
                model_format_name=None,
                model_format_version=None,
                version_description=None,
            ):
                mock_model = MagicMock()
                mock_model.name = name
                mock_model.version = version
                mock_model.uri = uri
                return mock_model

        mr_client = MockModelRegistryClient()

    mr_client.register_model(
        name=model_name,
        uri=trained_model_uri,
        version=version,
        model_format_name="pytorch",
        model_format_version="2.0",
        version_description="Model trained and versioned via orchestrated pipeline",
    )
    print("Model registration completed!")

## Step 3: Define and Chain the Pipeline DAG

We chain the components together by passing outputs as arguments and setting step execution orders.

In [ ]:
@dsl.pipeline(name="orchestrated-ml-pipeline")
def orchestrator_pipeline(
    input_path: str = "s3://raw-data",
    output_path: str = "s3://processed-data",
    model_name: str = "mnist-pipeline-model",
    version: str = "v1.0.0",
):
    # 1. Preprocess raw data
    preprocess_task = preprocess_data(input_path=input_path, output_path=output_path)

    # 2. Optimize learning rate
    tune_task = tune_hyperparameters(data_path=output_path)
    tune_task.after(preprocess_task)

    # 3. Train the final model with best learning rate
    train_task = train_final_model(best_lr=tune_task.output, data_path=output_path)
    train_task.after(tune_task)

    # 4. Register the model version
    register_task = register_model(
        trained_model_uri=train_task.outputs["trained_model_uri"],
        model_name=model_name,
        version=version,
    )
    register_task.after(train_task)

## Step 4: Run the Pipeline

Now we submit the pipeline using `PipelinesClient` and wait for execution to complete. If the client cannot connect to a running Kubeflow Pipelines backend, it falls back to executing the tasks sequentially for dry-run testing.

In [ ]:
from unittest.mock import MagicMock

try:
    print("Connecting to PipelinesClient...")
    pipelines_client = PipelinesClient()
    pipelines_client.list_pipelines()
    is_mock = False
    print("Successfully connected to PipelinesClient!")
except Exception as e:
    print(
        f"Could not connect to KFP PipelinesClient: {e}. Falling back to E2E sequential mock runner."
    )
    is_mock = True

if is_mock:
    # Local sequential test runner simulating pipelines execution
    class MockPipelinesClient:
        def run(self, pipeline, params=None):
            p_name = getattr(pipeline, "__name__", str(pipeline))
            print(f"[MOCK KFP] Compiling and executing pipeline: {p_name}")
            p = params or {}
            input_p = p.get("input_path", "s3://raw-data")
            output_p = p.get("output_path", "s3://processed-data")

            # Execute components in order using their unwrapped .python_func function bodies
            preprocess_data.python_func(input_path=input_p, output_path=output_p)
            best_lr = tune_hyperparameters.python_func(data_path=output_p)

            import tempfile

            temp_dir = tempfile.mkdtemp()
            model_path = os.path.join(temp_dir, "model_checkpoint_uri.txt")

            train_final_model.python_func(
                best_lr=best_lr, data_path=output_p, trained_model_uri=model_path
            )
            with open(model_path, "r") as f:
                trained_uri = f.read().strip()

            register_model.python_func(
                trained_model_uri=trained_uri,
                model_name=p.get("model_name", "mnist-pipeline-model"),
                version=p.get("version", "v1.0.0"),
            )

            mock_run = MagicMock()
            mock_run.name = p_name
            mock_run.id = "mock-pipeline-run-id-6789"
            mock_run.state = "Succeeded"
            return mock_run

        def wait_for_run_status(self, run, timeout=None):
            print(
                f"[MOCK KFP] Waiting for pipeline run {getattr(run, 'name', str(run))} to finish..."
            )
            return run

    pipelines_client = MockPipelinesClient()

# Execute the pipeline run
run = pipelines_client.run(
    orchestrator_pipeline,
    params={
        "input_path": "s3://my-raw-data-bucket",
        "output_path": "s3://my-processed-data-bucket",
        "model_name": "mnist-pipeline-model",
        "version": "v1.0.0",
    },
)
print(f"Pipeline run submitted successfully: {getattr(run, 'name', str(run))}")

# Wait for completion
completed_run = pipelines_client.wait_for_run_status(run)
print(
    f"Orchestrated pipeline run completed with state: {getattr(completed_run, 'state', 'Succeeded')}"
)